# PRUEBAS DE TABLAS FINALES GOLD

### 1. Ventas netas por sucursal y mes.

In [0]:
%sql
WITH ventas_mes AS (
    SELECT 
        EXTRACT(YEAR FROM fecha_venta) AS anho,
        EXTRACT(MONTH FROM fecha_venta) AS mes,
        sucursal_id,
        SUM(monto_total) AS ventas,
        COUNT(DISTINCT venta_id) AS cantidad_ventas
    FROM electrocasa_dev.silver.ventas
    WHERE fecha_venta >= '2026-03-01' 
      AND fecha_venta < '2026-11-01'
    GROUP BY 1, 2, 3
),
devoluciones_mes AS (
    SELECT 
        EXTRACT(YEAR FROM fecha_devolucion) AS anho,
        EXTRACT(MONTH FROM fecha_devolucion) AS mes,
        sucursal_id,
        SUM(monto_reembolso) AS devoluciones
    FROM electrocasa_dev.silver.devoluciones
    WHERE fecha_devolucion >= '2026-03-01' 
      AND fecha_devolucion < '2026-11-01'
    GROUP BY 1, 2, 3
)
SELECT 
    COALESCE(v.anho, d.anho) AS anho,
    COALESCE(v.mes, d.mes) AS mes,
    sc.sucursal_id,
    sc.nombre_sucursal,
    COALESCE(v.ventas, 0) AS ventas,
    COALESCE(d.devoluciones, 0) AS devoluciones,
    COALESCE(v.ventas, 0) - COALESCE(d.devoluciones, 0) AS ventas_netas,
    COALESCE(v.cantidad_ventas, 0) AS cantidad_ventas,
    ROUND(
        CASE 
            WHEN COALESCE(v.cantidad_ventas, 0) > 0 THEN v.ventas / v.cantidad_ventas 
            ELSE 0 
        END, 2
    ) AS ticket_promedio
FROM ventas_mes v
FULL OUTER JOIN devoluciones_mes d 
    ON v.anho = d.anho 
    AND v.mes = d.mes 
    AND v.sucursal_id = d.sucursal_id
INNER JOIN electrocasa_dev.silver.sucursales sc 
    ON sc.sucursal_id = COALESCE(v.sucursal_id, d.sucursal_id)
ORDER BY 1, 2, 3

### 2. Productos, sobre todo el período:

In [0]:
%sql
WITH ventas AS(
    SELECT ct.producto_id, ct.nombre_producto, SUM(vt.cantidad) cantidad FROM electrocasa_dev.silver.ventas vt
    INNER JOIN electrocasa_dev.silver.catalogo ct ON vt.producto_id = ct.producto_id
    WHERE vt.fecha_venta >= '2026-03-01' AND vt.fecha_venta < '2026-11-01'
    GROUP BY 1, 2
    
)
SELECT producto_id, nombre_producto, cantidad FROM ventas ORDER BY cantidad DESC LIMIT 10

In [0]:
%sql
WITH ventas AS(
    SELECT ct.producto_id, ct.nombre_producto, COUNT(DISTINCT venta_id) cantidad_venta  FROM electrocasa_dev.silver.ventas vt
    INNER JOIN electrocasa_dev.silver.catalogo ct ON vt.producto_id = ct.producto_id
    WHERE vt.fecha_venta >= '2026-03-01' AND vt.fecha_venta < '2026-11-01'
    GROUP BY 1, 2
),
ventas_producto AS(
    SELECT producto_id, nombre_producto, cantidad_venta FROM ventas
),
devoluciones AS(
    SELECT ct.producto_id, ct.nombre_producto, COUNT(DISTINCT devolucion_id) cantidad_dev FROM electrocasa_dev.silver.devoluciones dv
    INNER JOIN electrocasa_dev.silver.ventas vt ON dv.venta_id = vt.venta_id -- Se validó que cada devolución tiene su venta asociada
    INNER JOIN electrocasa_dev.silver.catalogo ct ON dv.producto_id = ct.producto_id
    WHERE fecha_devolucion >= '2026-03-01' AND fecha_devolucion < '2026-11-01'
    GROUP BY 1, 2
),
devoluciones_producto AS(
    SELECT producto_id, nombre_producto, cantidad_dev FROM devoluciones
),
vista_resumen AS(
    SELECT vp.producto_id, vp.nombre_producto, vp.cantidad_venta, COALESCE(dp.cantidad_dev, 0) AS cantidad_dev,ROUND(COALESCE(dp.cantidad_dev, 0) / vp.cantidad_venta, 4) AS ratio  FROM ventas_producto vp
    LEFT JOIN devoluciones_producto dp ON vp.producto_id = dp.producto_id
    ORDER BY vp.cantidad_venta DESC
)
SELECT producto_id, nombre_producto, ratio FROM vista_resumen ORDER BY ratio DESC LIMIT 10


### 3. Dotación

In [0]:
%sql
WITH fechas_base AS (
    SELECT 
        LAST_DAY(mes_inicio) AS ultimo_dia_mes
    FROM (
        SELECT EXPLODE(
            SEQUENCE(DATE '2026-03-01', DATE '2026-10-01', INTERVAL 1 MONTH)
        ) AS mes_inicio
    )
)
SELECT 
    EXTRACT(YEAR FROM fb.ultimo_dia_mes) AS anho,
    EXTRACT(MONTH FROM fb.ultimo_dia_mes) AS mes,
    fb.ultimo_dia_mes,
    s.sucursal_id,
    s.nombre_sucursal,
    COUNT(DISTINCT ep.id_empleado) AS cantidad_empleados_activos
FROM fechas_base fb
CROSS JOIN electrocasa_dev.silver.sucursales s
LEFT JOIN electrocasa_dev.silver.empleados ep
    ON ep.sucursal_id = s.sucursal_id
    AND ep.__START_AT < (fb.ultimo_dia_mes + INTERVAL 1 DAY)
    AND (ep.__END_AT >= (fb.ultimo_dia_mes + INTERVAL 1 DAY) OR ep.__END_AT IS NULL)
GROUP BY 1, 2, 3, 4, 5
ORDER BY fb.ultimo_dia_mes, s.sucursal_id

### 4. Ventas por vendedor

In [0]:
%sql
WITH ventas_base AS(
    SELECT venta_id, vendedor_id, cantidad, monto_total, fecha_venta FROM electrocasa_dev.silver.ventas
    WHERE vendedor_id IS NOT NULL AND canal = 'TIENDA_FISICA'
),
empleados as(
    SELECT ep.id_empleado, ep.nombre, ep.sucursal_id, s.nombre_sucursal, ep.__START_AT, ep.__END_AT FROM electrocasa_dev.silver.empleados ep
    INNER JOIN electrocasa_dev.silver.sucursales s ON s.sucursal_id = ep.sucursal_id
),
vista_cruzada AS(
    SELECT EXTRACT(YEAR FROM vb.fecha_venta) anho, EXTRACT(MONTH FROM vb.fecha_venta) mes,vb.venta_id, vb.vendedor_id,
    CASE
        WHEN ep.nombre IS NULL THEN 'SIN_ASIGNAR'
        ELSE ep.nombre
    END nombre
    , vb.cantidad, vb.monto_total, ep.sucursal_id, 
    CASE
        WHEN ep.nombre_sucursal IS NULL THEN 'SIN_ASIGNAR'
        ELSE ep.nombre_sucursal
    END sucursal
    FROM ventas_base vb
    LEFT JOIN empleados ep ON vb.vendedor_id = ep.id_empleado AND vb.fecha_venta >= ep.__START_AT AND (vb.fecha_venta < ep.__END_AT OR ep.__END_AT IS NULL)
),
vista_resultado AS(
    SELECT 
    anho, 
    mes,
    vendedor_id,
    nombre,
    sucursal_id,
    sucursal,
    COUNT(venta_id) nro_ventas,
    SUM(cantidad) cantidad,
    SUM(monto_total) monto_total
    FROM vista_cruzada
    GROUP BY 1, 2, 3, 4, 5, 6
    ORDER BY 1, 2, 3, 4, 5, 6
)
SELECT * FROM vista_resultado 

### 5. Reseñas

In [0]:
%sql
WITH vista_total AS(
    SELECT
    CONCAT(
        EXTRACT(YEAR FROM rs.fecha_resena),
        '-',
        EXTRACT(MONTH FROM rs.fecha_resena),
        '-',
        ct.categoria
    ) ID,
    ct.categoria,
    EXTRACT(YEAR FROM rs.fecha_resena) anho,
    EXTRACT(MONTH FROM rs.fecha_resena) mes,
    COUNT(rs.resena_id) num_resenas_total
    FROM electrocasa_dev.silver.resenas rs
    INNER JOIN electrocasa_dev.silver.catalogo ct ON rs.producto_id = ct.producto_id
    GROUP BY 1,2,3,4
    ORDER BY 1,2,3,4
),
vista_2 AS(
    SELECT
    CONCAT(
        EXTRACT(YEAR FROM rs.fecha_resena),
        '-',
        EXTRACT(MONTH FROM rs.fecha_resena),
        '-',
        ct.categoria
    ) ID,
    EXTRACT(YEAR FROM rs.fecha_resena) anho,
    EXTRACT(MONTH FROM rs.fecha_resena) mes,
    ct.categoria,
    CASE
        WHEN rs.calificacion IN (1, 2) THEN 'NEGATIVO'
        ELSE 'POSITIVO'
    END categoria_calificacion,
    COUNT(rs.resena_id) num_resenas
    FROM electrocasa_dev.silver.resenas rs
    INNER JOIN electrocasa_dev.silver.catalogo ct ON rs.producto_id = ct.producto_id
    GROUP BY 1,2,3,4,5
    ORDER BY 1,2,3,4,5
),
vista_cruzada AS(
    SELECT
        v2.anho,
        v2.mes,
        v2.categoria,
        v2.categoria_calificacion,
        v2.num_resenas,
        vt.num_resenas_total,
        v2.num_resenas / vt.num_resenas_total * 100 porcentaje
    FROM vista_2 v2
    LEFT JOIN vista_total vt ON v2.ID = vt.ID
    ORDER BY 1,2,3,4
),
vista_resultado AS(
    SELECT anho, mes, categoria, categoria_calificacion, porcentaje FROM vista_cruzada WHERE categoria_calificacion = 'NEGATIVO'
)
SELECT * FROM vista_resultado

### 6. Despachos

In [0]:
%sql
WITH eventos_limpios AS (
    SELECT 
        tracking_id,
        courier,
        estado_entrega,
        fecha_modificacion
    FROM electrocasa_dev.silver.tracking_envios
    WHERE fecha_modificacion >= '2026-03-01' 
      AND fecha_modificacion < '2026-11-01'
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY tracking_id, estado_entrega, fecha_modificacion 
        ORDER BY evento_id
    ) = 1
),
resumen_envio AS (
    SELECT 
        tracking_id,
        FIRST_VALUE(courier) AS courier,
        MIN(CASE WHEN estado_entrega = 'PENDIENTE' THEN fecha_modificacion END) AS fecha_pendiente,
        MIN(CASE WHEN estado_entrega = 'ENTREGADO' THEN fecha_modificacion END) AS fecha_entregado,
        MAX(CASE WHEN estado_entrega = 'DEVUELTO' THEN 1 ELSE 0 END) AS es_devuelto,
        MAX(CASE WHEN estado_entrega IN ('ENTREGADO', 'DEVUELTO') THEN 1 ELSE 0 END) AS tiene_estado_final
    FROM eventos_limpios
    GROUP BY tracking_id
)
SELECT 
    courier,
    ROUND(
        AVG(
            CASE 
                WHEN fecha_entregado IS NOT NULL AND fecha_pendiente IS NOT NULL 
                THEN TIMESTAMPDIFF(SECOND, fecha_pendiente, fecha_entregado) / 3600.0 
            END
        ), 2
    ) AS tiempo_promedio_horas,
    ROUND(
        SUM(es_devuelto) * 100.0 / NULLIF(COUNT(*), 0), 2
    ) AS porcentaje_devueltos,
    COUNT(*) AS total_envios_finalizados,
    SUM(es_devuelto) AS total_envios_devueltos
FROM resumen_envio
WHERE tiene_estado_final = 1
GROUP BY courier
ORDER BY courier

In [0]:
%sql
DESCRIBE FUNCTION EXTENDED electrocasa_dev.silver.mask_salario